# KV Cache: 键值缓存

KV Cache是自回归生成中的关键优化技术，通过缓存已计算的Key和Value来避免重复计算。

## 核心概念

### 为什么需要KV Cache？

在自回归生成中，每个新token都需要关注之前的所有token：

```
Step 1: "The"         → 计算 K₁, V₁
Step 2: "The cat"     → 计算 K₁, V₁, K₂, V₂  (K₁, V₁重复计算！)
Step 3: "The cat sat" → 计算 K₁, V₁, K₂, V₂, K₃, V₃  (更多重复！)
```

**KV Cache的解决方案：**
- 缓存已计算的K和V
- 每步只计算新token的K、V
- 拼接到缓存中使用

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 1. KVCache类实现

In [ ]:
class KVCache:
    """KV缓存实现"""
    
    def __init__(self, num_heads, head_dim, max_seq_len=2048):
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        
        # 预分配缓存空间
        self.k_cache = np.zeros((num_heads, max_seq_len, head_dim))
        self.v_cache = np.zeros((num_heads, max_seq_len, head_dim))
        
        self.cache_len = 0
    
    def update(self, new_k, new_v):
        """更新缓存"""
        new_seq_len = new_k.shape[1]
        
        # 添加到缓存
        self.k_cache[:, self.cache_len:self.cache_len + new_seq_len, :] = new_k
        self.v_cache[:, self.cache_len:self.cache_len + new_seq_len, :] = new_v
        
        self.cache_len += new_seq_len
        
        # 返回完整的K、V
        return self.k_cache[:, :self.cache_len, :], self.v_cache[:, :self.cache_len, :]
    
    def reset(self):
        """重置缓存"""
        self.cache_len = 0

# 测试
kv_cache = KVCache(num_heads=8, head_dim=64, max_seq_len=100)

# 第一次更新（Prefill）
k1 = np.random.randn(8, 10, 64)  # 10个prompt tokens
v1 = np.random.randn(8, 10, 64)
full_k, full_v = kv_cache.update(k1, v1)

print(f"第一次更新后:")
print(f"  缓存长度: {kv_cache.cache_len}")
print(f"  K形状: {full_k.shape}")

# 第二次更新（Decode）
k2 = np.random.randn(8, 1, 64)  # 1个新token
v2 = np.random.randn(8, 1, 64)
full_k, full_v = kv_cache.update(k2, v2)

print(f"\n第二次更新后:")
print(f"  缓存长度: {kv_cache.cache_len}")
print(f"  K形状: {full_k.shape}")

## 2. 计算量对比可视化

In [ ]:
# 不同序列长度下的计算量对比
seq_lengths = np.array([1, 10, 50, 100, 200, 500, 1000])

# 无缓存：O(n²)
no_cache_ops = seq_lengths ** 2

# 有缓存：O(n) per step
with_cache_ops = seq_lengths + 1

# 绘图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 绝对计算量
ax1.plot(seq_lengths, no_cache_ops, 'r-o', label='No Cache (O(n²))', linewidth=2)
ax1.plot(seq_lengths, with_cache_ops, 'b-s', label='With Cache (O(n))', linewidth=2)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Number of Operations')
ax1.set_title('Computation Cost: No Cache vs With Cache')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 加速比
speedup = no_cache_ops / with_cache_ops
ax2.plot(seq_lengths, speedup, 'g-o', linewidth=2, markersize=8)
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Speedup (x)')
ax2.set_title('KV Cache Speedup vs Sequence Length')
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

print("计算量对比表:")
print(f"{'长度':<8} {'无缓存':<15} {'有缓存':<15} {'加速比':<10}")
print("-" * 50)
for sl, no_c, with_c, sp in zip(seq_lengths, no_cache_ops, with_cache_ops, speedup):
    print(f"{sl:<8} {no_c:<15,} {with_c:<15,} {sp:<10.1f}x")

## 3. 带KV Cache的多头注意力

In [ ]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

class MultiHeadAttentionWithKVCache:
    """带KV Cache的多头注意力"""
    
    def __init__(self, embed_dim, num_heads, max_seq_len=2048):
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # 投影矩阵
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        # KV缓存
        self.kv_cache = KVCache(num_heads, self.head_dim, max_seq_len)
    
    def split_heads(self, x):
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 0, 2)
    
    def combine_heads(self, x):
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        return x.reshape(seq_len, self.embed_dim)
    
    def forward_with_cache(self, x, use_cache=True, is_prefill=False):
        """带缓存的前向传播"""
        # 计算Q, K, V
        Q = np.dot(x, self.W_q.T)
        K_new = np.dot(x, self.W_k.T)
        V_new = np.dot(x, self.W_v.T)
        
        # 分割头
        Q = self.split_heads(Q)
        K_new = self.split_heads(K_new)
        V_new = self.split_heads(V_new)
        
        # 更新缓存
        if use_cache:
            K_full, V_full = self.kv_cache.update(K_new, V_new)
        else:
            K_full, V_full = K_new, V_new
        
        # 计算注意力
        outputs = []
        for i in range(self.num_heads):
            scores = np.dot(Q[i], K_full[i].T) / np.sqrt(self.head_dim)
            
            # Causal mask for decode
            if use_cache and not is_prefill:
                new_seq_len, total_len = scores.shape
                mask = np.tril(np.ones((new_seq_len, total_len)))
                scores = np.where(mask == 0, -1e9, scores)
            
            attn_weights = softmax(scores, axis=-1)
            output_i = np.dot(attn_weights, V_full[i])
            outputs.append(output_i)
        
        # 合并头
        multi_head_output = np.stack(outputs, axis=0)
        concatenated = self.combine_heads(multi_head_output)
        
        # 输出投影
        return np.dot(concatenated, self.W_o.T)

# 测试
mha = MultiHeadAttentionWithKVCache(embed_dim=512, num_heads=8)

# Prefill
prompt = np.random.randn(20, 512)
output_prefill = mha.forward_with_cache(prompt, use_cache=True, is_prefill=True)

print(f"Prefill:")
print(f"  输入: {prompt.shape}")
print(f"  输出: {output_prefill.shape}")
print(f"  缓存长度: {mha.kv_cache.cache_len}")

# Decode
new_token = np.random.randn(1, 512)
output_decode = mha.forward_with_cache(new_token, use_cache=True, is_prefill=False)

print(f"\nDecode:")
print(f"  输入: {new_token.shape}")
print(f"  输出: {output_decode.shape}")
print(f"  缓存长度: {mha.kv_cache.cache_len}")

## 4. 自回归生成流程可视化

In [ ]:
# 模拟自回归生成过程
def autoregressive_generation_demo(prompt_len=10, num_new_tokens=5):
    """演示自回归生成过程"""
    mha_demo = MultiHeadAttentionWithKVCache(embed_dim=64, num_heads=4, max_seq_len=100)
    
    # Prefill
    prompt = np.random.randn(prompt_len, 64)
    _ = mha_demo.forward_with_cache(prompt, use_cache=True, is_prefill=True)
    
    cache_lengths = [mha_demo.kv_cache.cache_len]
    
    # Decode
    for i in range(num_new_tokens):
        new_token = np.random.randn(1, 64)
        _ = mha_demo.forward_with_cache(new_token, use_cache=True, is_prefill=False)
        cache_lengths.append(mha_demo.kv_cache.cache_len)
    
    return cache_lengths

# 运行演示
cache_lens = autoregressive_generation_demo(prompt_len=10, num_new_tokens=15)

# 可视化
plt.figure(figsize=(12, 6))
steps = list(range(len(cache_lens)))
plt.plot(steps, cache_lens, 'o-', linewidth=2, markersize=8)
plt.axvline(x=0, color='r', linestyle='--', alpha=0.5, label='Prefill')
plt.axvspan(1, len(cache_lens)-1, alpha=0.1, color='blue', label='Decode')
plt.xlabel('Generation Step')
plt.ylabel('Cache Length (tokens)')
plt.title('KV Cache Growth During Autoregressive Generation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"生成过程:")
print(f"  Prompt长度: {cache_lens[0]}")
print(f"  生成新tokens: {len(cache_lens) - 1}")
print(f"  最终序列长度: {cache_lens[-1]}")

## 5. KV Cache显存占用分析

In [ ]:
# 计算不同配置下的KV Cache显存占用
def calculate_kv_cache_memory(num_layers, num_heads, head_dim, seq_len, batch_size=1, dtype_bytes=4):
    """计算KV Cache显存占用（GB）"""
    # 每层: 2 (K和V) × num_heads × seq_len × head_dim × dtype_bytes
    per_layer_bytes = 2 * num_heads * seq_len * head_dim * dtype_bytes
    total_bytes = num_layers * per_layer_bytes * batch_size
    return total_bytes / (1024 ** 3)  # 转换为GB

# 不同模型配置
configs = [
    ('GPT-2 Small', 12, 12, 64, 1024),
    ('GPT-2 Medium', 24, 16, 64, 1024),
    ('GPT-2 Large', 36, 20, 64, 1024),
    ('LLaMA-7B', 32, 32, 128, 2048),
    ('LLaMA-13B', 40, 40, 128, 2048),
    ('GPT-3 175B', 96, 96, 128, 2048),
]

# 计算不同batch size的显存占用
batch_sizes = [1, 4, 8, 16]
results = {}

for name, layers, heads, head_dim, seq_len in configs:
    results[name] = [calculate_kv_cache_memory(layers, heads, head_dim, seq_len, bs) 
                     for bs in batch_sizes]

# 可视化
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(batch_sizes))
width = 0.12

for i, (name, values) in enumerate(results.items()):
    offset = (i - len(results)/2) * width
    ax.bar(x + offset, values, width, label=name)

ax.set_xlabel('Batch Size')
ax.set_ylabel('KV Cache Memory (GB)')
ax.set_title('KV Cache Memory Usage by Model and Batch Size')
ax.set_xticks(x)
ax.set_xticklabels(batch_sizes)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

# 打印表格
print("\nKV Cache显存占用（GB）:")
print(f"{'模型':<15} {'BS=1':<10} {'BS=4':<10} {'BS=8':<10} {'BS=16':<10}")
print("-" * 60)
for name, values in results.items():
    print(f"{name:<15} {values[0]:<10.2f} {values[1]:<10.2f} {values[2]:<10.2f} {values[3]:<10.2f}")

## 6. Prefill vs Decode阶段对比

In [ ]:
# 对比Prefill和Decode阶段的特性
comparison_data = {
    'Prefill': {
        'Input': 'Prompt (多个tokens)',
        'Compute': 'O(prompt_len²)',
        'KV Cache': '初始化并填充',
        'Bottleneck': '计算密集',
        'Parallelism': '高（可并行）'
    },
    'Decode': {
        'Input': '1个新token',
        'Compute': 'O(total_len)',
        'KV Cache': '增量更新',
        'Bottleneck': '显存带宽',
        'Parallelism': '低（序列化）'
    }
}

# 创建对比表格
import pandas as pd

df = pd.DataFrame(comparison_data).T
print("\nPrefill vs Decode 阶段对比:")
print(df.to_string())

# 时间占比可视化
# 假设生成100个tokens，prompt长度50
prompt_len = 50
num_generated = 100

# 估算相对时间（简化）
prefill_time = prompt_len ** 2  # O(n²)
decode_time_per_step = prompt_len + np.arange(num_generated)  # 逐步增长
total_decode_time = np.sum(decode_time_per_step)

times = [prefill_time, total_decode_time]
labels = ['Prefill', 'Decode (100 tokens)']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 饼图
ax1.pie(times, labels=labels, autopct='%1.1f%%', startangle=90)
ax1.set_title('Time Distribution (Prompt=50, Generate=100)')

# Decode时间随生成长度变化
gen_lengths = np.arange(1, 201)
decode_times = [np.sum(prompt_len + np.arange(gl)) for gl in gen_lengths]

ax2.plot(gen_lengths, decode_times, linewidth=2)
ax2.set_xlabel('Number of Generated Tokens')
ax2.set_ylabel('Cumulative Decode Time')
ax2.set_title('Decode Time Growth')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. KV Cache优化技术

### Multi-Query Attention (MQA)
所有头共享同一个K、V，显存减少num_heads倍。

### Grouped-Query Attention (GQA)
多个头共享一组K、V，平衡效果和显存。LLaMA-2采用。

### 量化
使用int8/int4存储K、V，显存减少2-4倍。

In [ ]:
# 不同优化技术的显存对比
num_heads = 32
head_dim = 128
seq_len = 2048
num_layers = 32

def kv_memory_gb(num_kv_heads, dtype_bytes=4):
    """计算KV Cache显存（GB）"""
    bytes_total = 2 * num_kv_heads * seq_len * head_dim * dtype_bytes * num_layers
    return bytes_total / (1024 ** 3)

techniques = {
    'Standard (FP32)': kv_memory_gb(num_heads, 4),
    'Standard (FP16)': kv_memory_gb(num_heads, 2),
    'MQA (FP16)': kv_memory_gb(1, 2),  # 单个K、V
    'GQA-8 (FP16)': kv_memory_gb(num_heads // 4, 2),  # 4头共享1组
    'Standard + INT8': kv_memory_gb(num_heads, 1),
    'MQA + INT8': kv_memory_gb(1, 1),
}

# 可视化
plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(techniques)), list(techniques.values()), 
               color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c'])
plt.xticks(range(len(techniques)), list(techniques.keys()), rotation=45, ha='right')
plt.ylabel('Memory (GB)')
plt.title(f'KV Cache Memory with Different Optimizations\n({num_layers} layers, {num_heads} heads, seq_len={seq_len})')
plt.grid(axis='y', alpha=0.3)

# 添加数值标签
for bar, val in zip(bars, techniques.values()):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.2f}GB',
            ha='center', va='bottom')

plt.tight_layout()
plt.show()

# 节省比例
baseline = techniques['Standard (FP32)']
print("\n显存节省（相对于Standard FP32）:")
for name, mem in techniques.items():
    savings = (1 - mem/baseline) * 100
    print(f"  {name:<20}: {mem:.2f}GB ({savings:+.1f}%)")

## 8. 总结

### KV Cache的关键特性

1. **性能提升**: 推理速度提升2-3倍（长序列更明显）
2. **计算优化**: 从O(n²)降到O(n)
3. **显存换速度**: 需要额外显存存储K、V
4. **两阶段**: Prefill（初始化）+ Decode（增量更新）

### 优化技术

- **MQA**: 共享K、V，显存最小
- **GQA**: 平衡效果和显存
- **量化**: INT8/INT4存储
- **Paged Attention**: 动态内存管理

### 实际应用

- 所有现代LLM都使用KV Cache
- GPT-3/4、LLaMA、Claude等
- 推理优化的关键技术